# Analysis Notebook
Analysis for run: nnodes100_prob0.20_noise1.00gaussian_ntargets1_true/seed8

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path

# Set working directory to the notebook's location
notebook_dir = Path.cwd()
print(f"Working directory: {notebook_dir}")

## Load Data

In [ ]:
# Load pickle files
# Try to load attrib_scores.pkl, fall back to attrib_scores_est.pkl
try:
    with open('attrib_scores.pkl', 'rb') as f:
        attrib_scores = pickle.load(f)
except FileNotFoundError:
    attrib_scores = None

with open('attrib_scores_est.pkl', 'rb') as f:
    attrib_scores_est = pickle.load(f)

with open('ground_truth_dag.pkl', 'rb') as f:
    ground_truth_dag = pickle.load(f)

with open('mechanism_changes.pkl', 'rb') as f:
    mechanism_changes = pickle.load(f)

with open('fitted_model_source.pkl', 'rb') as f:
    fitted_model = pickle.load(f)

# Load CSV files
data = pd.read_csv('data.csv')
data_new = pd.read_csv('data_new.csv')
solutions_comb = pd.read_csv('solutions_comb.csv')
solutions_reg = pd.read_csv('solutions_reg.csv')
solutions_reg_processed = pd.read_csv('solutions_reg_processed.csv')
solutions_single = pd.read_csv('solutions_single.csv')

# Load ground truth interventions
with open('ground_truth_interventions.txt', 'r') as f:
    ground_truth_interventions = f.read()

print("Data loaded successfully")

## Compute Metrics

### (1) Data Statistics

In [3]:
# Compute statistics for data.csv
data_stats = {}

# Data mean: mean of the means of all columns
data_stats['data_mean'] = data.mean().mean()

# Data std: mean of the standard deviations of all columns
data_stats['data_std'] = data.std().mean()

# Data range: minimum and maximum value of the entire data matrix
data_stats['data_min'] = data.min().min()
data_stats['data_max'] = data.max().max()

# Compute statistics for data_new.csv
data_new_stats = {}

# Data mean: mean of the means of all columns
data_new_stats['data_new_mean'] = data_new.mean().mean()

# Data std: mean of the standard deviations of all columns
data_new_stats['data_new_std'] = data_new.std().mean()

# Data range: minimum and maximum value of the entire data matrix
data_new_stats['data_new_min'] = data_new.min().min()
data_new_stats['data_new_max'] = data_new.max().max()

# Combine all statistics
metrics = {**data_stats, **data_new_stats}

print("Data Statistics:")
print("\ndata.csv:")
for key in data_stats:
    print(f"  {key}: {data_stats[key]}")
print("\ndata_new.csv:")
for key in data_new_stats:
    print(f"  {key}: {data_new_stats[key]}")

Data Statistics:

data.csv:
  data_mean: 0.0009838555682104028
  data_std: 0.8258199198932619
  data_min: -3.649169846521441
  data_max: 4.58081940843689

data_new.csv:
  data_new_mean: -0.016689202188784617
  data_new_std: 0.8303756099881298
  data_new_min: -4.9850147755217264
  data_new_max: 5.1155060484369965


### (2) Variable Ranking

In [4]:
import re
import os

# Extract seed from folder path
folder_name = os.path.basename(notebook_dir)
seed_match = re.search(r'seed(\d+)', folder_name)
seed = int(seed_match.group(1)) if seed_match else 42
np.random.seed(seed)
print(f"Using seed: {seed}")

# Get all variable names from data
all_variables = list(data.columns)
num_variables = len(all_variables)
print(f"Total number of variables: {num_variables}")
print(f"Variables: {all_variables}")

# Stage 1: Rank based on frequency in solutions_reg_processed.csv
# First, deduplicate solutions by intervention targets
solutions_deduplicated = solutions_reg_processed.drop_duplicates(subset=['intervention targets'])
print(f"Original solutions: {len(solutions_reg_processed)}, Unique solutions: {len(solutions_deduplicated)}")

# Count occurrences of each variable in intervention targets (after deduplication)
variable_counts = {var: 0 for var in all_variables}

for idx, row in solutions_deduplicated.iterrows():
    intervention_targets = row['intervention targets']
    # Parse the intervention targets string
    # Format is like "Index(['X5'], dtype='object')"
    if 'Index' in str(intervention_targets):
        # Extract variable names from the Index string
        matches = re.findall(r"'([^']+)'", intervention_targets)
        for var in matches:
            if var in variable_counts:
                variable_counts[var] += 1

print(f"\nVariable frequencies in intervention targets (after deduplication):")
for var, count in sorted(variable_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {var}: {count}")

# Sort variables by count (descending) and randomly shuffle ties
stage1_vars = [var for var, count in variable_counts.items() if count > 0]
stage1_counts = [variable_counts[var] for var in stage1_vars]

# Group by count and shuffle within each group
from collections import defaultdict
count_groups = defaultdict(list)
for var in stage1_vars:
    count_groups[variable_counts[var]].append(var)

stage1_ranked = []
for count in sorted(count_groups.keys(), reverse=True):
    group = count_groups[count]
    np.random.shuffle(group)
    stage1_ranked.extend(group)

print(f"\nStage 1: {len(stage1_ranked)} variables ranked based on intervention frequency")

# Stage 2: Rank remaining variables by attribution scores
# Load attribution scores (use attrib_scores.pkl if available, else attrib_scores_est.pkl)
attrib_scores_dict = attrib_scores if attrib_scores is not None else attrib_scores_est

stage2_vars = [var for var in all_variables if var not in stage1_ranked and var in attrib_scores_dict]

# Sort by attribution score (descending) and shuffle ties
stage2_scores = [(var, attrib_scores_dict[var]) for var in stage2_vars]
score_groups = defaultdict(list)
for var, score in stage2_scores:
    score_groups[score].append(var)

stage2_ranked = []
for score in sorted(score_groups.keys(), reverse=True):
    group = score_groups[score]
    np.random.shuffle(group)
    stage2_ranked.extend(group)

print(f"Stage 2: {len(stage2_ranked)} variables ranked based on attribution scores")

# Stage 3: Randomly rank all remaining variables
stage3_vars = [var for var in all_variables if var not in stage1_ranked and var not in stage2_ranked]
np.random.shuffle(stage3_vars)

print(f"Stage 3: {len(stage3_vars)} variables randomly ranked")

# Combine all stages
final_ranking = stage1_ranked + stage2_ranked + stage3_vars

# Create rank dictionary (1-indexed)
variable_ranks = {var: rank + 1 for rank, var in enumerate(final_ranking)}

# Verify all variables are ranked
assert len(variable_ranks) == num_variables, "Not all variables have ranks!"
assert set(variable_ranks.values()) == set(range(1, num_variables + 1)), "Ranks are not unique or have gaps!"

print(f"\nFinal ranking complete: all {num_variables} variables ranked from 1 to {num_variables}")

# Save to CSV
ranking_df = pd.DataFrame({
    'Variable': final_ranking,
    'Rank': [variable_ranks[var] for var in final_ranking],
    'Stage': (['Stage1_Intervention'] * len(stage1_ranked) + 
              ['Stage2_Attribution'] * len(stage2_ranked) + 
              ['Stage3_Random'] * len(stage3_vars))
})

# Add frequency and attribution score for reference
ranking_df['Intervention_Count'] = ranking_df['Variable'].map(variable_counts)
ranking_df['Attribution_Score'] = ranking_df['Variable'].map(attrib_scores_dict)

ranking_df.to_csv('variable_rankings.csv', index=False)
print("\nVariable rankings saved to 'variable_rankings.csv'")

# Display top 10
print("\nTop 10 ranked variables:")
print(ranking_df.head(10).to_string(index=False))

Using seed: 8
Total number of variables: 100
Variables: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X38', 'X39', 'X40', 'X41', 'X42', 'X43', 'X44', 'X45', 'X46', 'X47', 'X48', 'X49', 'X50', 'X51', 'X52', 'X53', 'X54', 'X55', 'X56', 'X57', 'X58', 'X59', 'X60', 'X61', 'X62', 'X63', 'X64', 'X65', 'X66', 'X67', 'X68', 'X69', 'X70', 'X71', 'X72', 'X73', 'X74', 'X75', 'X76', 'X77', 'X78', 'X79', 'X80', 'X81', 'X82', 'X83', 'X84', 'X85', 'X86', 'X87', 'X88', 'X89', 'X90', 'X91', 'X92', 'X93', 'X94', 'X95', 'X96', 'X97', 'X98', 'X99', 'X100']
Original solutions: 20, Unique solutions: 1

Variable frequencies in intervention targets (after deduplication):
  X11: 1
  X1: 0
  X2: 0
  X3: 0
  X4: 0
  X5: 0
  X6: 0
  X7: 0
  X8: 0
  X9: 0
  X10: 0
  X12: 0
  X13: 0
  X14: 0
  X15: 0
  X16: 0
  X

### (3) Recall at k

In [5]:
# Parse ground truth interventions
gt_interventions = []
with open('ground_truth_interventions.txt', 'r') as f:
    for line in f:
        value = float(line.strip())
        gt_interventions.append(value)

# Identify true targets (non-zero interventions)
# Row index (0-based) + 1 gives variable number (e.g., row 0 -> X1)
true_targets = []
for idx, value in enumerate(gt_interventions):
    if value != 0.0:
        var_name = f'X{idx + 1}'
        true_targets.append(var_name)

num_true_targets = len(true_targets)
print(f"Number of true targets (a_size): {num_true_targets}")
print(f"True target variables: {true_targets}")

# Extract a_size from parent folder name to verify
parent_folder = os.path.dirname(notebook_dir)
parent_name = os.path.basename(parent_folder)
a_size_match = re.search(r'ntargets(\d+)', parent_name)
a_size_from_folder = int(a_size_match.group(1)) if a_size_match else None
print(f"a_size from folder name: {a_size_from_folder}")

# Verify they match
if a_size_from_folder is not None:
    assert num_true_targets == a_size_from_folder, f"Mismatch: {num_true_targets} != {a_size_from_folder}"
    print("✓ a_size matches folder name")

# Set k = a_size
k = num_true_targets

# Get top k variables from our ranking
top_k_vars = final_ranking[:k]
print(f"\nTop {k} ranked variables: {top_k_vars}")

# Calculate recall@k: fraction of top k that are true targets
correct_in_top_k = [var for var in top_k_vars if var in true_targets]
recall_at_k = len(correct_in_top_k) / k if k > 0 else 0.0

print(f"\nCorrect targets in top {k}: {correct_in_top_k}")
print(f"Recall@{k}: {recall_at_k:.4f} ({len(correct_in_top_k)}/{k})")

# Add to metrics
metrics['a_size'] = a_size_from_folder
metrics['num_true_targets'] = num_true_targets
metrics['recall_at_k'] = recall_at_k
metrics['true_targets'] = true_targets

# If recall@k != 1, check ranks of all true targets
if recall_at_k != 1.0:
    print(f"\n⚠ Recall@{k} is not perfect (≠ 1.0)")
    true_target_ranks = {var: variable_ranks[var] for var in true_targets}
    print("Ranks of all true targets:")
    for var, rank in sorted(true_target_ranks.items(), key=lambda x: x[1]):
        in_top_k = "✓" if rank <= k else "✗"
        print(f"  {var}: rank {rank} {in_top_k}")
    
    # Save the ranks
    metrics['true_target_ranks'] = true_target_ranks
else:
    print(f"\n✓ Perfect recall@{k}! All true targets are in top {k}")
    metrics['true_target_ranks'] = {var: variable_ranks[var] for var in true_targets}

Number of true targets (a_size): 1
True target variables: ['X11']
a_size from folder name: 1
✓ a_size matches folder name

Top 1 ranked variables: ['X11']

Correct targets in top 1: ['X11']
Recall@1: 1.0000 (1/1)

✓ Perfect recall@1! All true targets are in top 1


### (4) Best Solution Analysis

In [6]:
# Find solution with # of intervention targets closest to a_size
solutions_reg_processed['target_count_diff'] = abs(
    solutions_reg_processed['# of intervention targets'] - num_true_targets
)

# Get minimum difference
min_diff = solutions_reg_processed['target_count_diff'].min()
closest_solutions = solutions_reg_processed[
    solutions_reg_processed['target_count_diff'] == min_diff
]

print(f"Looking for solutions with # of intervention targets closest to {num_true_targets}")
print(f"Found {len(closest_solutions)} solution(s) with difference of {min_diff}")

# If multiple solutions, select the one with highest TransProp
best_solution = closest_solutions.loc[closest_solutions['TransProp'].idxmax()]

print(f"\nBest solution:")
print(f"  # of intervention targets: {best_solution['# of intervention targets']}")
print(f"  TransProp: {best_solution['TransProp']:.6f}")
print(f"  Lambda: {best_solution['Lambda']:.4f}")
print(f"  Diff: {best_solution['Diff']:.6f}")

# Save these values
metrics['best_solution_num_targets'] = int(best_solution['# of intervention targets'])
metrics['best_solution_transprop'] = best_solution['TransProp']

# Parse intervention targets and values
intervention_targets_str = best_solution['intervention targets']
intervention_values_str = best_solution['intervention values']

# Extract variable names from intervention targets (filter to only X* pattern)
all_matches = re.findall(r"'([^']+)'", intervention_targets_str)
predicted_targets = [m for m in all_matches if m.startswith('X')]

# Extract intervention values (parse list)
# Extract intervention values (parse numpy array string format)
if intervention_values_str == '[]':
    predicted_values = []
else:
    # Remove brackets and split by whitespace to handle numpy array format
    values_str = intervention_values_str.strip('[]')
    predicted_values = [float(x) for x in values_str.split()]

print(f"\nPredicted intervention targets: {predicted_targets}")
print(f"Predicted intervention values: {predicted_values}")

# Create a dictionary mapping variable to predicted intervention value
predicted_interventions = {var: val for var, val in zip(predicted_targets, predicted_values)}

# Ground truth interventions as dictionary (variable -> true value)
true_interventions_dict = {}
for idx, value in enumerate(gt_interventions):
    var_name = f'X{idx + 1}'
    true_interventions_dict[var_name] = value

# Calculate absolute differences for all variables
intervention_value_differences = {}
for var in all_variables:
    predicted_val = predicted_interventions.get(var, 0.0)  # 0 if not intervened
    true_val = true_interventions_dict.get(var, 0.0)
    abs_diff = abs(predicted_val - true_val)
    intervention_value_differences[var] = abs_diff

print(f"\nAbsolute differences between predicted and true intervention values:")
for var in sorted(intervention_value_differences.keys(), 
                   key=lambda x: int(x[1:])):  # Sort by variable number
    diff = intervention_value_differences[var]
    pred = predicted_interventions.get(var, 0.0)
    true = true_interventions_dict.get(var, 0.0)
    if diff > 0 or true != 0 or pred != 0:  # Only show non-zero cases
        print(f"  {var}: |{pred:.6f} - {true:.6f}| = {diff:.6f}")

# Save the differences
metrics['intervention_value_differences'] = intervention_value_differences

# Calculate summary statistics
total_abs_error = sum(intervention_value_differences.values())
error_per_true_target = total_abs_error / num_true_targets

print(f"\nSummary:")
print(f"  Total absolute error divided by # of true targets: {error_per_true_target:.6f}")

metrics['error_per_true_target'] = error_per_true_target

Looking for solutions with # of intervention targets closest to 1
Found 20 solution(s) with difference of 0

Best solution:
  # of intervention targets: 1
  TransProp: 0.999954
  Lambda: 0.0045
  Diff: 0.009319

Predicted intervention targets: ['X11']
Predicted intervention values: [1.22367422]

Absolute differences between predicted and true intervention values:
  X11: |1.223674 - 1.230288| = 0.006614

Summary:
  Total absolute error divided by # of true targets: 0.006614


### (5) DE Analysis Comparison

In [7]:
# (1) Read DE_results.csv and calculate recall@k using DE ranking
de_results = pd.read_csv('DE_results.csv', index_col=0)

# The index of de_results is the ranking (first index = rank 1, second = rank 2, etc.)
de_ranking = list(de_results.index)

# Calculate recall@k for DE (k = num_true_targets = a_size)
k_de = num_true_targets
de_top_k = de_ranking[:k_de]
de_correct_in_top_k = [var for var in de_top_k if var in true_targets]
de_recall_at_k = len(de_correct_in_top_k) / k_de if k_de > 0 else 0.0

print(f"\nDE top {k_de} variables: {de_top_k}")
print(f"Correct targets in DE top {k_de}: {de_correct_in_top_k}")
print(f"DE Recall@{k_de}: {de_recall_at_k:.4f} ({len(de_correct_in_top_k)}/{k_de})")

# True target ranks by DE (1-indexed)
de_variable_ranks = {var: rank + 1 for rank, var in enumerate(de_ranking)}
de_true_target_ranks = {var: de_variable_ranks[var] for var in true_targets if var in de_variable_ranks}
print(f"\nTrue target ranks by DE: {de_true_target_ranks}")

# Save to metrics
metrics['de_recall_at_k'] = de_recall_at_k
metrics['de_true_target_ranks'] = de_true_target_ranks


DE top 1 variables: ['X43']
Correct targets in DE top 1: []
DE Recall@1: 0.0000 (0/1)

True target ranks by DE: {'X11': 57}


In [8]:
# (2) Read solutions_DE_supp.csv and get the solution with same # of intervention targets as best solution
solutions_DE_supp = pd.read_csv('solutions_DE_supp.csv')

best_solution_num_targets = int(best_solution['# of intervention targets'])

# Get DE solution with same number of intervention targets as best solution
de_matching = solutions_DE_supp[
    solutions_DE_supp['# of intervention targets'] == best_solution_num_targets
]

if len(de_matching) == 0:
    print(f"No DE solution found with {best_solution_num_targets} intervention targets")
    de_best_transprop = None
else:
    de_best_solution = de_matching.iloc[0]
    de_best_transprop = de_best_solution['TransProp']
    print(f"\nDE solution with {best_solution_num_targets} intervention targets:")
    print(f"  TransProp: {de_best_transprop:.6f}")
    print(f"  intervention targets: {de_best_solution['intervention targets']}")

# Save to metrics
metrics['de_best_transprop'] = de_best_transprop
print(f"\nDE best TransProp saved: {de_best_transprop}")


DE solution with 1 intervention targets:
  TransProp: 0.145826
  intervention targets: ['X43']

DE best TransProp saved: 0.1458259694505349


In [9]:
# (3) Plot comparison of our solutions vs DE solutions (TransProp vs # of intervention targets)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numpy import trapz

# Prepare our solutions: remove duplicate # of intervention targets, keep highest TransProp
our_solutions = solutions_reg_processed.copy()
our_solutions = our_solutions.sort_values('TransProp', ascending=False)
our_solutions = our_solutions.drop_duplicates(subset=['# of intervention targets'], keep='first')
our_solutions = our_solutions.sort_values('# of intervention targets')

# Prepend (0, 0) to both lines
our_x = [0] + list(our_solutions['# of intervention targets'])
our_y = [0] + list(our_solutions['TransProp'])

# Prepare DE solutions
de_solutions = solutions_DE_supp.copy()
de_solutions = de_solutions.sort_values('# of intervention targets')
de_x = [0] + list(de_solutions['# of intervention targets'])
de_y = [0] + list(de_solutions['TransProp'])

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(our_x, our_y, marker='o', label='Our Solutions')
ax.plot(de_x, de_y, marker='s', label='DE Solutions')
ax.set_xlabel('# of intervention targets')
ax.set_ylabel('TransProp')
ax.set_title('Our Solutions vs DE Solutions')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('solutions_comparison.png', dpi=300)
plt.close()
print("Plot saved to 'solutions_comparison.png'")

# Calculate area under the two lines (using trapezoidal rule)
auc_our = trapz(our_y, our_x)
auc_de = trapz(de_y, de_x)

print(f"\nArea under our solutions curve: {auc_our:.6f}")
print(f"Area under DE solutions curve:  {auc_de:.6f}")

# Save to metrics
metrics['auc_our_solutions'] = auc_our
metrics['auc_de_solutions'] = auc_de

Plot saved to 'solutions_comparison.png'

Area under our solutions curve: 0.499977
Area under DE solutions curve:  0.072913


In [10]:
# (4) Calculate average TransProp difference across # of intervention targets
# For each # of intervention targets (excluding 0), compute our TransProp - DE TransProp
# Then average these differences

# Build lookup: # of intervention targets -> best TransProp for our solutions
our_tp_lookup = dict(zip(our_solutions['# of intervention targets'], our_solutions['TransProp']))
# Build lookup: # of intervention targets -> TransProp for DE solutions
de_tp_lookup = dict(zip(de_solutions['# of intervention targets'], de_solutions['TransProp']))

# Get all unique # of intervention targets present in EITHER set (excluding 0)
all_num_targets = sorted(set(list(our_tp_lookup.keys()) + list(de_tp_lookup.keys())))
all_num_targets = [n for n in all_num_targets if n > 0]

print(f"# of intervention targets (excluding 0): {all_num_targets}")
print()

transprop_diffs = []
for n in all_num_targets:
    our_tp = our_tp_lookup.get(n, 0.0)
    de_tp = de_tp_lookup.get(n, 0.0)
    diff = our_tp - de_tp
    transprop_diffs.append(diff)
    print(f"  # targets={n}: Our TransProp={our_tp:.6f}, DE TransProp={de_tp:.6f}, Diff={diff:.6f}")

avg_transprop_diff = np.mean(transprop_diffs)
print(f"\nAverage TransProp difference (Our - DE): {avg_transprop_diff:.6f}")

# Save to metrics
metrics['avg_transprop_diff'] = avg_transprop_diff


# of intervention targets (excluding 0): [1]

  # targets=1: Our TransProp=0.999954, DE TransProp=0.145826, Diff=0.854128

Average TransProp difference (Our - DE): 0.854128


In [11]:
# (5) Calculate average TransProp for our method and DE separately
# Average across all # of intervention targets (excluding 0)

our_transprops = [our_tp_lookup.get(n, 0.0) for n in all_num_targets]
de_transprops = [de_tp_lookup.get(n, 0.0) for n in all_num_targets]

avg_transprop_our = np.mean(our_transprops)
avg_transprop_de = np.mean(de_transprops)

print(f"Average TransProp (Our):  {avg_transprop_our:.6f}")
print(f"Average TransProp (DE):   {avg_transprop_de:.6f}")

# Save to metrics
metrics['avg_transprop_our'] = avg_transprop_our
metrics['avg_transprop_de'] = avg_transprop_de


Average TransProp (Our):  0.999954
Average TransProp (DE):   0.145826


## Save Results

In [12]:
# Save metrics to JSON
with open('analysis_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Results saved to analysis_metrics.json")

Results saved to analysis_metrics.json
